# Keystone-Σ₀ PLT — Colab test harness

Loads, verifies, and (optionally) faithfully-parity-checks the proprietary Σ₀ PLT coder we own
([ADR-0011](https://github.com/alex-place/lantern-os/blob/master/docs/adr/0011-proprietary-sigma0-base-model.md)).
It clones the **public** repo and runs the *real shipped pipeline* — nothing is reimplemented here.

**Pick a GPU runtime:** Runtime ▸ Change runtime type ▸ GPU.

| Runtime | VRAM | What you get |
|---|---|---|
| T4 (free) | 16 GB | 4-bit load + smoke + (slow) HumanEval subset |
| L4 / A100 (Pro) | 24 / 40 GB | **bf16** load (clean, no quant noise) + faithful vLLM parity |

Already validated on an 8 GB RTX 3070 at 4-bit: weight-key match `missing=0 unexpected=0`, 5.71 GB,
generates correct Python. Colab adds the bf16 + faithful-parity rungs the 8 GB box can't reach.

## 1 · GPU + pick dtype

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU.'
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
# 7.6B bf16 ~15GB weights; needs ~22GB+ with the 2-loop activations. Else 4-bit.
DTYPE = 'bf16' if VRAM >= 22 else '4bit'
print(f'VRAM {VRAM:.0f} GB  ->  dtype = {DTYPE}')

## 2 · Dependencies
Torch ships with Colab. Pin transformers to the version the model config targets.

In [ ]:
%pip -q install 'transformers==4.57.1' 'huggingface_hub>=0.25' 'accelerate>=1.0' \
               'bitsandbytes>=0.44' 'peft>=0.13' sentencepiece safetensors datasets

## 3 · Clone the (public) repo
`GIT_LFS_SKIP_SMUDGE=1` skips the LFS image objects (the repo's LFS budget is capped and we need none of them) — without it the checkout aborts on a budget error.

In [ ]:
import os
if not os.path.isdir('/content/lantern-os'):
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/alex-place/lantern-os /content/lantern-os
%cd /content/lantern-os/models/keystone-sigma0-plt
!git -C /content/lantern-os rev-parse --short HEAD

## 4 · Download + patch the checkpoint
Pulls the Apache-2.0 LoopCoder-V2 weights (~18 GB) and wires our `auto_map`. One time (~few min on Colab).

In [ ]:
!python download_and_patch.py --out /content/ckpt

## 5 · Stage-0 parity
Weight-key match + single-forward next-token + short generations + peak VRAM. On an A100/L4 this runs in **bf16** — the cleanest functional check short of vLLM logit parity.

In [ ]:
!python check_parity.py --model /content/ckpt --dtype $DTYPE --max-new-tokens 64

## 6 · (Optional) HumanEval functional signal
Does the ported forward generate code that **passes real unit tests**? A strong functional proof (not faithful parity). Generation has no KV cache yet (full recompute), so this is slow — default 10 problems. Bump `N` on an A100.

In [ ]:
N = 10  # number of HumanEval problems (start small; generation is O(n^2) without KV cache)
import torch, signal, contextlib, io
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

kw = dict(trust_remote_code=True)
if DTYPE == '4bit':
    kw.update(quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
              bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True), device_map='cuda:0')
else:
    kw.update(torch_dtype=torch.bfloat16, device_map='cuda:0')
tok = AutoTokenizer.from_pretrained('/content/ckpt', trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained('/content/ckpt', **kw).eval()

ds = load_dataset('openai_humaneval')['test'].select(range(N))

def stop_at_def(text):
    # crude truncation: stop at the next top-level def/class or end marker
    for marker in ['\ndef ', '\nclass ', '\nif __name__', '\nprint(', '\n@']:
        i = text.find(marker)
        if i != -1: text = text[:i]
    return text

def run_with_timeout(src, secs=5):
    def handler(*a): raise TimeoutError()
    old = signal.signal(signal.SIGALRM, handler); signal.alarm(secs)
    try:
        env = {}
        with contextlib.redirect_stdout(io.StringIO()):
            exec(src, env); return True
    except Exception as e:
        return False
    finally:
        signal.alarm(0); signal.signal(signal.SIGALRM, old)

passed = 0
for ex in ds:
    ids = tok(ex['prompt'], return_tensors='pt', return_token_type_ids=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=256, do_sample=False)
    comp = stop_at_def(tok.decode(out[0][ids['input_ids'].shape[-1]:], skip_special_tokens=True))
    prog = ex['prompt'] + comp + '\n' + ex['test'] + f"\ncheck({ex['entry_point']})\n"
    ok = run_with_timeout(prog)
    passed += ok
    print(f"{ex['task_id']:15} {'PASS' if ok else 'fail'}")
print(f'\npass@1 (greedy) on {N} problems: {passed}/{N} = {passed/N:.0%}')

## 7 · (Optional, A100) Definitive faithful parity vs the vLLM fork
The real Stage-0 gate: `top1_agree ≥ 0.99` between our HF forward and the vendor's vLLM reference on identical inputs. Heavy — builds the fork from source (long), needs ≥24 GB.

```bash
# 1. build the vendor fork (slow):
pip install git+https://github.com/yxing-bj/vllm
# 2. dump reference logits for a fixed input_ids -> ref_logits.pt  ({'input_ids','logits'})
#    (run the LoopCoder-V2 model under vLLM, capture last-step logits [1,T,V])
# 3. compare:
python check_parity.py --model /content/ckpt --dtype bf16 --ref ref_logits.pt
```
If `top1_agree < 0.99`, the suspect is one of the three reconstructed boundaries (CLP shift / sliding-window edge / per-loop norm) — all config-gated in `modeling_keystone_plt.py`. This cell is intentionally manual: building vLLM forks in Colab is finicky and not worth automating until someone commits to the A100 run.